# 00_0 — Orphan Detection

**Purpose:** Find tracks that were listened to but never landed in any playlist or liked song. Save their URIs so they can be added to Spotify, exported via Exportify, and enriched with audio features in `00_1`.

**Runs before:** `00_1_build_full_history.ipynb`.

## Inputs / Outputs

| Input | Path | Format |
|---|---|---|
| Listening history | `data/Spotify/Spotify_ Extended_Streaming_History/**/Streaming_History_Audio_*.json` | JSON |
| Exportify playlists | `data/Exportify/spotify_playlists (*)/*.csv` | CSV |
| Spotify credentials (optional) | `spotify_credentials.json` | JSON |

| Output | Path | Description |
|---|---|---|
| `orphans_uris.csv` | `data/Exportify/orphans_to_add/` | All orphan URIs |
| `orphans_batch_NN.csv` | `data/Exportify/orphans_to_add/` | Batches of 10,000 (Spotify playlist limit) |
| `created_playlists.json` | `data/Exportify/orphans_to_add/` | Playlist metadata (API mode only) |

## Limitations

- **N=1** — one profile only, numbers don't transfer.
- **Premium required for API mode** — free account returns 403, use `'manual'`.
- **Exportify is the only manual step** — no API.
- **No atomicity** — API crash mid-batch → conflict on rerun, delete playlist manually.
- **7 of 18 742 without `Valence`** — Exportify didn't return them; `00_1` puts them in `orphan_category='unknown'`.
- **Orphans depend on Exportify export moment** — order: export → `00_0` → don't touch playlists → `00_1`.

## Cell map

| # | Section |
|---|---------|
| 0 | Imports, paths, folder structure |
| 0.5 | Spotify API credentials |
| 1 | Unique tracks from listening history |
| 2 | Unique tracks from Exportify playlists |
| 3 | Comparison |
| 4 | Save orphan lists |
| 5 | Method branch (`api` / `manual`) |
| 6A | Create playlists via API |
| 6B | Manual instructions |
| 7 | Final check |

## 0. Imports, paths, folder structure

In [1]:
# 0. IMPORTS, PATHS, FOLDER STRUCTURE
# ============================================================================
# Step 0 of the pipeline: detect tracks that are not in any playlist or liked
# songs. Save the list of orphan URIs for manual Spotify upload.
#
# Runs BEFORE 00_1_build_full_history.ipynb.
# ============================================================================

import sys
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd


def find_src(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'src').is_dir() and (p / '01_data_analysis').is_dir():
            return p / 'src'
    raise FileNotFoundError('src/ not found above current directory')


SRC_PATH = find_src()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from paths import get_data_path, get_project_root

# ---------- Base paths ----------
PROJECT_ROOT = get_project_root()
DATA = get_data_path()

# ---------- Full structure ----------
PATHS = {
    'streaming_history': DATA / 'Spotify' / 'Spotify_ Extended_Streaming_History',
    'account_data':      DATA / 'Spotify' / 'Spotify_Account_ Data',
    'exportify':         DATA / 'Exportify',
    'orphan_export':     DATA / 'Exportify' / 'exportify_orphan_export',
    'orphans_to_add':    DATA / 'Exportify' / 'orphans_to_add',
    'output':            DATA / '00_output',
}

# ---------- Create folders if missing ----------
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# ---------- Glob patterns: nested + flat fallback ----------
def _find(nested_pattern, flat_pattern):
    files = sorted(glob.glob(nested_pattern))
    if not files:
        files = sorted(glob.glob(flat_pattern))
    return files

history_files = _find(
    str(PATHS['streaming_history'] / '*' / 'Streaming_History_Audio_*.json'),
    str(PATHS['streaming_history'] / 'Streaming_History_Audio_*.json'),
)

exportify_files = _find(
    str(PATHS['exportify'] / '*' / 'spotify_playlists (*)' / '*.csv'),
    str(PATHS['exportify'] / 'spotify_playlists (*)' / '*.csv'),
)

orphan_files = sorted(glob.glob(str(PATHS['orphan_export'] / '*.csv')))

# Account Data JSON counter
json_nested = sorted(glob.glob(str(PATHS['account_data'] / '*' / '*.json')))
json_flat = sorted(glob.glob(str(PATHS['account_data'] / '*.json')))
account_json_files = json_nested if json_nested else json_flat

# ---------- Constants ----------
BATCH_SIZE = 10_000

# ---------- Status ----------
print('=' * 78)
print('DATA STRUCTURE')
print('=' * 78)
print()
print(f'  Streaming History      : {len(history_files)} files')
print(f'  Account Data           : {len(account_json_files)} JSON files')
print(f'  Exportify playlists    : {len(exportify_files)} CSV files')
print(f'  Orphan export          : {len(orphan_files)} CSV files')
print()
print('=' * 78)
print('WHAT TO PUT WHERE')
print('=' * 78)
print()
print('  1. data/Spotify/Spotify_ Extended_Streaming_History/')
print('       Streaming_History_Audio_*.json')
print()
print('  2. data/Spotify/Spotify_Account_ Data/')
print('       YourLibrary.json, Playlist*.json, UserAttributes.json, ...')
print()
print('  3. data/Exportify/spotify_playlists (<account>)/')
print('       <playlist>.csv, Liked_Songs.csv')
print()
print('  4. data/Exportify/exportify_orphan_export/')
print('       <orphan>.csv (filled later, after Exportify)')
print()
print('  5. data/Exportify/orphans_to_add/')
print('       orphans_uris.csv, orphans_batch_NN.csv (created by this notebook)')
print()
print('  6. data/00_output/')
print('       intermediate parquet files (created by 00_1 / 00_2)')
print()

if len(history_files) == 0:
    print('WARNING: No history JSON found.')
if len(exportify_files) == 0:
    print('WARNING: No Exportify playlist CSV found.')
if len(history_files) > 0 and len(exportify_files) > 0:
    print('Ready. Run the next cell.')

DATA STRUCTURE

  Streaming History      : 15 files
  Account Data           : 38 JSON files
  Exportify playlists    : 364 CSV files
  Orphan export          : 8 CSV files

WHAT TO PUT WHERE

  1. data/Spotify/Spotify_ Extended_Streaming_History/
       Streaming_History_Audio_*.json

  2. data/Spotify/Spotify_Account_ Data/
       YourLibrary.json, Playlist*.json, UserAttributes.json, ...

  3. data/Exportify/spotify_playlists (<account>)/
       <playlist>.csv, Liked_Songs.csv

  4. data/Exportify/exportify_orphan_export/
       <orphan>.csv (filled later, after Exportify)

  5. data/Exportify/orphans_to_add/
       orphans_uris.csv, orphans_batch_NN.csv (created by this notebook)

  6. data/00_output/
       intermediate parquet files (created by 00_1 / 00_2)

Ready. Run the next cell.


## 0.5. Spotify API credentials

In [2]:
# 0.5. SPOTIFY API CREDENTIALS
# ============================================================================
# Loads client_id / client_secret / redirect_uri from a local JSON file.
# The file is ignored by git and never committed.
# ============================================================================

CREDS_PATH = Path.cwd() / 'spotify_credentials.json'
if not CREDS_PATH.exists():
    CREDS_PATH = Path(SRC_PATH).parent / '01_data_analysis' / '00_prep' / 'spotify_credentials.json'

if not CREDS_PATH.exists():
    print('WARNING: spotify_credentials.json not found.')
    print('   Create the file next to the notebook.')
    SPOTIFY = None
else:
    with open(CREDS_PATH, 'r', encoding='utf-8') as f:
        SPOTIFY = json.load(f)

    missing = [k for k in ('client_id', 'client_secret', 'redirect_uri') if not SPOTIFY.get(k)]
    if missing:
        print(f'WARNING: missing keys in credentials: {missing}')
        SPOTIFY = None
    else:
        print('Spotify credentials loaded.')

Spotify credentials loaded.


## 1. Unique tracks from listening history

In [3]:
# 1. UNIQUE TRACKS FROM LISTENING HISTORY
# ============================================================================

frames = []
for f in history_files:
    df = pd.read_json(f)
    df['account'] = Path(f).parent.name
    frames.append(df)

df_hist = pd.concat(frames, ignore_index=True)

df_unique = df_hist.drop_duplicates(subset=['spotify_track_uri'])[
    ['spotify_track_uri', 'master_metadata_track_name', 'master_metadata_album_artist_name']
].rename(columns={
    'master_metadata_track_name': 'track_name',
    'master_metadata_album_artist_name': 'artist_name',
})

df_unique = df_unique.dropna(subset=['spotify_track_uri'])
df_unique['spotify_track_uri'] = df_unique['spotify_track_uri'].astype(str)
df_unique = df_unique[df_unique['spotify_track_uri'].str.strip() != '']

history_uris = set(df_unique['spotify_track_uri'])

print(f'Raw events:          {len(df_hist):,}')
print(f'Unique tracks:       {len(df_unique):,}')

Raw events:          80,055
Unique tracks:       25,629


## 2. Unique tracks from Exportify playlists

In [4]:
# 2. UNIQUE TRACKS FROM EXPORTIFY PLAYLISTS
# ============================================================================

playlist_uris = set()
n_files = 0

for f in exportify_files:
    try:
        df = pd.read_csv(f, usecols=['Track URI'])
        playlist_uris.update(df['Track URI'].dropna().astype(str))
        n_files += 1
    except Exception:
        pass

playlist_uris = {u for u in playlist_uris if u.strip() != ''}

print(f'CSV files:           {n_files}')
print(f'Unique tracks:       {len(playlist_uris):,}')

CSV files:           364
Unique tracks:       24,953


## 3. Comparison

In [5]:
# 3. COMPARISON
# ============================================================================
# Three groups:
#   1. In history AND in playlists   - listened, saved to playlists
#   2. Only in history               - ORPHANS
#   3. Only in playlists             - saved but never listened
# ============================================================================

in_both   = history_uris & playlist_uris
only_hist = history_uris - playlist_uris     # ORPHANS
only_pl   = playlist_uris - history_uris

print('=' * 70)
print('COMPARISON')
print('=' * 70)
print()
print(f'  History:                     {len(history_uris):,}')
print(f'  Playlists:                   {len(playlist_uris):,}')
print()
print(f'  1. In history AND playlists: {len(in_both):,}')
print(f'  2. Only in history (ORPHANS):{len(only_hist):,}')
print(f'  3. Only in playlists:        {len(only_pl):,}')
print()
print(f'  Check: {len(in_both):,} + {len(only_hist):,} = {len(in_both) + len(only_hist):,}')

COMPARISON

  History:                     25,629
  Playlists:                   24,953

  1. In history AND playlists: 7,234
  2. Only in history (ORPHANS):18,395
  3. Only in playlists:        17,719

  Check: 7,234 + 18,395 = 25,629


## 4. Save orphan lists

In [6]:
# 4. SAVE ORPHAN LISTS
# ============================================================================
# Saves:
#   orphans_uris.csv       - all orphans in one file
#   orphans_batch_NN.csv   - batches of 10,000 (Spotify playlist limit)
# ============================================================================

only_hist = {u for u in only_hist if u and str(u).strip() != ''}

ORPHANS_OUT = PATHS['orphans_to_add']
orphans_sorted = sorted(only_hist)

pd.DataFrame({'spotify_track_uri': orphans_sorted}).to_csv(
    ORPHANS_OUT / 'orphans_uris.csv', index=False
)
print(f'  orphans_uris.csv       : {len(orphans_sorted):,}')

n_batches = (len(orphans_sorted) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(n_batches):
    chunk = orphans_sorted[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
    pd.DataFrame({'spotify_track_uri': chunk}).to_csv(
        ORPHANS_OUT / f'orphans_batch_{i+1:02d}.csv', index=False
    )
    print(f'  orphans_batch_{i+1:02d}.csv : {len(chunk):,}')

print()
print(f'  Total orphans:   {len(orphans_sorted):,}')
print(f'  Batches:         {n_batches}')
print(f'  Output folder:   data/Exportify/orphans_to_add/')

  orphans_uris.csv       : 18,395
  orphans_batch_01.csv : 10,000
  orphans_batch_02.csv : 8,395

  Total orphans:   18,395
  Batches:         2
  Output folder:   data/Exportify/orphans_to_add/


## 5. Method branch

In [ ]:
# 5. BRANCH: 'api' OR 'manual' 
# ============================================================================
# METHOD = 'manual'
METHOD = 'api'   # 'api' or 'manual'

## 6A. Create playlists via Spotify API

In [ ]:
# 6A. CREATE ORPHAN PLAYLISTS VIA SPOTIFY API
# ============================================================================
# Requires:
#   - spotify_credentials.json next to the notebook
#   - Spotify Premium account
#   - OAuth (browser opens once)
#
# Creates N private playlists: __ORPHANS_01, __ORPHANS_02, ...
# Adds tracks in batches of 100 (API limit).
# Saves metadata to orphans_to_add/created_playlists.json
# ============================================================================

if METHOD == 'api':

    if SPOTIFY is None:
        print('No Spotify credentials. Check spotify_credentials.json and cell 0.5.')

    else:
        import spotipy
        from spotipy.oauth2 import SpotifyOAuth

        # ---------- OAuth ----------
        sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
            client_id=SPOTIFY['client_id'],
            client_secret=SPOTIFY['client_secret'],
            redirect_uri=SPOTIFY['redirect_uri'],
            scope='playlist-modify-private playlist-read-private',
            cache_path='.spotify_cache',
        ))

        me = sp.me()
        print('Spotify authorization OK.')
        print()

        # ---------- Existing playlists (pagination) ----------
        existing = {}
        offset = 0
        while True:
            page = sp._get('me/playlists', limit=50, offset=offset)
            for pl in page['items']:
                if pl and pl.get('name'):
                    existing[pl['name']] = pl
            if page.get('next') is None:
                break
            offset += 50

        # ---------- Duplicate protection ----------
        created = []
        skipped = []
        conflict = []

        for i in range(n_batches):
            chunk = orphans_sorted[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
            pl_name = f'__ORPHANS_{i+1:02d}'

            if pl_name in existing:
                pl = existing[pl_name]
                info = sp._get(f'playlists/{pl["id"]}/items', limit=1)
                n_existing = info['total']

                if n_existing == len(chunk):
                    print(f'  {pl_name}: already exists ({n_existing:,}) - skipping')
                    skipped.append({
                        'name': pl_name,
                        'id': pl['id'],
                        'url': pl['external_urls']['spotify'],
                        'n_tracks': n_existing,
                        'status': 'skipped',
                    })
                    continue
                else:
                    print(f'  {pl_name}: exists with {n_existing:,}, expected {len(chunk):,}')
                    conflict.append({
                        'name': pl_name,
                        'existing_id': pl['id'],
                        'existing_n': n_existing,
                        'expected_n': len(chunk),
                    })
                    continue

            # ---------- Create ----------
            print(f'  {pl_name}: {len(chunk):,} tracks')
            pl = sp._post('me/playlists', payload={
                'name': pl_name,
                'public': False,
                'description': f'Orphan tracks batch {i+1}/{n_batches}',
            })
            pid = pl['id']

            for j in range(0, len(chunk), 100):
                sp._post(f'playlists/{pid}/items', payload={'uris': chunk[j : j + 100]})

            print(f'     added {len(chunk):,}')
            created.append({
                'name': pl_name,
                'id': pid,
                'url': pl['external_urls']['spotify'],
                'n_tracks': len(chunk),
                'status': 'created',
            })

        # ---------- Conflicts ----------
        if conflict:
            print()
            print('=' * 70)
            print('CONFLICTS (same name, different track count)')
            print('=' * 70)
            for c in conflict:
                print(f"  {c['name']}: in Spotify {c['existing_n']:,}, expected {c['expected_n']:,}")
            print()
            print('  Options:')
            print('    A. Delete these playlists in Spotify manually, then re-run.')
            print('    B. Rename them in Spotify.')
            print('    C. Leave as is - these batches will not be created.')
            print()

        # ---------- Metadata ----------
        meta_path = ORPHANS_OUT / 'created_playlists.json'
        meta = created + skipped
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

        # ---------- Summary ----------
        print()
        print('=' * 70)
        print(f'  Created:    {len(created)}')
        print(f'  Skipped:    {len(skipped)} (already existed)')
        print(f'  Conflicts:  {len(conflict)}')
        print(f'  Metadata:   data/Exportify/orphans_to_add/created_playlists.json')
        print('=' * 70)
        print()
        print('NEXT STEPS:')
        print('  1. Open https://exportify.net/ and authorize with Spotify')
        print('  2. Find playlists __ORPHANS_01, __ORPHANS_02, ...')
        print('  3. Export each - downloads CSV with audio features')
        print('  4. Put CSVs into data/Exportify/exportify_orphan_export/')
        print('  5. Run 00_1_build_full_history.ipynb')

Spotify authorization OK.

  __ORPHANS_01: 10,000 tracks
     added 10,000
  __ORPHANS_02: 8,395 tracks


## 6B. Manual instructions

In [ ]:
# 6b. MANUAL PLAYLIST CREATION
# ============================================================================

if METHOD == 'manual':

    print('=' * 78)
    print('MANUAL ORPHAN PLAYLIST CREATION')
    print('=' * 78)
    print()
    print(f'  Orphans total:    {len(orphans_sorted):,}')
    print(f'  Batch files:      {n_batches}')
    print(f'  Location:         data/Exportify/orphans_to_add/')
    print()
    print('=' * 78)
    print('STEP 1. CREATE PRIVATE PLAYLISTS IN SPOTIFY')
    print('=' * 78)
    print()
    print('  Open Spotify DESKTOP APP.')
    print('  WARNING: the web version does NOT support bulk URI paste.')
    print()
    print('  Create one playlist per batch file. Names (exact):')
    print()
    for i in range(n_batches):
        n_in_batch = min(BATCH_SIZE, len(orphans_sorted) - i * BATCH_SIZE)
        print(f'      __ORPHANS_{i+1:02d}      ({n_in_batch:,} tracks)')
    print()
    print('  IMPORTANT:')
    print('    - Set each playlist to PRIVATE (three dots -> Make private)')
    print('    - Otherwise orphans appear in your public profile')
    print()
    print('=' * 78)
    print('STEP 2. ADD TRACKS')
    print('=' * 78)
    print()
    print('  Work in the DESKTOP app, not in the browser.')
    print()
    print('  For each file orphans_batch_NN.csv:')
    print('    1. Open the CSV in Excel or a text editor')
    print('    2. Copy the entire spotify_track_uri column')
    print('    3. Open the matching playlist in Spotify')
    print('    4. Inside the playlist press Ctrl+V')
    print('       (or three dots -> Add to this playlist -> paste)')
    print('    5. Confirm')
    print()
    print('  Spotify accepts the whole batch in a single paste.')
    print('  Repeat for all batches.')
    print()
    print('  Check: playlist track count = number of rows in the CSV.')
    print()
    print('=' * 78)
    print('STEP 3. EXPORT VIA EXPORTIFY')
    print('=' * 78)
    print()
    print('  1. Open https://exportify.net/')
    print('  2. Log in with Spotify (one time)')
    print('  3. Find playlists __ORPHANS_01, __ORPHANS_02, ... in the list')
    print('  4. Click Export on each -> a CSV with audio features downloads')
    print()
    print('=' * 78)
    print('STEP 4. PUT CSV FILES IN PLACE')
    print('=' * 78)
    print()
    print('  Move all downloaded CSVs into:')
    print('      data/Exportify/exportify_orphan_export/')
    print()
    print('  Expected: same number of files as batch files.')
    print()
    print('=' * 78)
    print('STEP 5. RUN THE NEXT NOTEBOOK')
    print('=' * 78)
    print()
    print('  Run 00_1_build_full_history.ipynb.')
    print('  It will pick up the CSV files as orphan references')
    print('  and enrich the history with audio features.')
    print()
    print('=' * 78)

## 7. Final check

In [ ]:
# 7. FINAL CHECK
# ============================================================================
# Shows:
#   1. How many orphans were detected in this run
#   2. How many of them were already exported (from exportify_orphan_export/)
#   3. How many exported tracks have audio features
# ============================================================================

print('=' * 70)
print('FINAL CHECK')
print('=' * 70)
print()

# ---------- Orphans from this run ----------
orphans_file = ORPHANS_OUT / 'orphans_uris.csv'
if orphans_file.exists():
    df_orphans = pd.read_csv(orphans_file)
    orphans_uris = set(df_orphans['spotify_track_uri'].astype(str))
    print(f'  Orphans (this run)         : {len(orphans_uris):,}')
else:
    orphans_uris = set()
    print('  Orphans (this run)         : 0 (file not created)')

# ---------- Already exported ----------
orphan_export_dir = PATHS['orphan_export']
exported_uris = set()
all_exported = []

if orphan_export_dir.exists():
    for f in sorted(orphan_export_dir.glob('*.csv')):
        try:
            df = pd.read_csv(f)
            if 'Track URI' in df.columns:
                exported_uris.update(df['Track URI'].dropna().astype(str))
                all_exported.append(df)
        except Exception:
            pass

exported_uris = {u for u in exported_uris if u.strip() != ''}
print(f'  Exported via Exportify     : {len(exported_uris):,}')
print()

# ---------- Coverage ----------
if len(orphans_uris) > 0 and len(exported_uris) > 0:
    covered = orphans_uris & exported_uris
    not_covered = orphans_uris - exported_uris
    print('=' * 70)
    print('COVERAGE')
    print('=' * 70)
    print(f'  Total orphans              : {len(orphans_uris):,}')
    print(f'  Covered by export          : {len(covered):,} ({len(covered)/len(orphans_uris)*100:.1f}%)')
    print(f'  Still to export            : {len(not_covered):,}')
    print()
    if len(not_covered) == 0:
        print('  All orphans exported. Ready to run 00_1.')
    else:
        print('  Some orphans are not exported yet.')

elif len(orphans_uris) > 0 and len(exported_uris) == 0:
    print('=' * 70)
    print('COVERAGE')
    print('=' * 70)
    print(f'  Orphans found              : {len(orphans_uris):,}')
    print(f'  But exportify_orphan_export/ is empty.')
    print()
    print('  Steps:')
    print('    1. Upload orphans to Spotify (API or manual)')
    print('    2. Export them via Exportify')
    print('    3. Put CSVs into data/Exportify/exportify_orphan_export/')
    print('    4. Run 00_1_build_full_history.ipynb')

elif len(orphans_uris) == 0:
    print('  No orphans detected. Check cells 1-4.')

print()
print('=' * 70)

# ---------- Audio features ----------
if all_exported:
    print()
    print('=' * 70)
    print('AUDIO FEATURES IN EXPORT')
    print('=' * 70)

    df_exported = pd.concat(all_exported, ignore_index=True)
    df_unique_exp = df_exported.drop_duplicates(subset=['Track URI']).copy()

    valence_col = None
    for c in ['Valence', 'valence']:
        if c in df_unique_exp.columns:
            valence_col = c
            break

    if valence_col is None:
        print('  Valence column not found in CSV.')
        print(f'  Columns: {df_unique_exp.columns.tolist()}')
    else:
        with_features = df_unique_exp[valence_col].notna().sum()
        without_features = df_unique_exp[valence_col].isna().sum()
        total = len(df_unique_exp)
        pct = with_features / total * 100 if total > 0 else 0

        print(f'  Tracks in export           : {total:,}')
        print(f'  With features (valence)    : {with_features:,} ({pct:.1f}%)')
        print(f'  Without features           : {without_features:,}')
        print()

        if pct >= 99:
            print('  All orphans have features.')
        elif pct >= 80:
            print(f'  Most have features ({pct:.1f}%).')
        elif pct >= 50:
            print(f'  Only {pct:.1f}% have features. Acceptable but not ideal.')
        else:
            print(f'  Low feature coverage ({pct:.1f}%). Check export.')

    print()
    print('=' * 70)

FINAL CHECK

  Orphans (this run)         : 18,395
  Exported via Exportify     : 18,742

COVERAGE
  Total orphans              : 18,395
  Covered by export          : 18,395 (100.0%)
  Still to export            : 0

  All orphans exported. Ready to run 00_1.


AUDIO FEATURES IN EXPORT
  Tracks in export           : 18,742
  With features (valence)    : 18,735 (100.0%)
  Without features           : 7

  All orphans have features.



<details>
<summary>🇷🇺 Итог</summary>

# 00_0 — Поиск сирот

Находит треки, которые слушались, но не попали ни в плейлисты, ни в лайки. Exportify отдаёт аудио-фичи только для треков в плейлистах, поэтому сирот надо сначала положить в плейлист.

**Запускается перед:** `00_1_build_full_history.ipynb`.

## Как запускать

1. **Клонировать репо и открыть ноут.** Ноут должен лежать внутри проекта — иначе `get_project_root()` не найдёт `src/`.
2. **Ячейка 0.** Создаёт всю структуру папок в `data/` (`Spotify/`, `Exportify/`, `00_output/` и подпапки) и печатает, что куда класть. Можно запустить **первой** — до того, как разложены файлы, — чтобы получить готовое дерево.
3. **Разложить файлы** по созданным папкам (см. «Входы» ниже).
4. **Ячейки 0.5–4.** Прочитать историю + Exportify, посчитать сирот, сохранить `orphans_uris.csv` и батчи по 10 000 в `data/Exportify/orphans_to_add/`.
5. **Ячейка 5.** Выбрать `METHOD`:
   - `'api'` — автоматически через Spotify API
   - `'manual'` — пошаговая инструкция
6. **Ячейка 6a (api)** или **6b (manual).** Создать приватные плейлисты `__ORPHANS_NN`.
7. **Ручной шаг (всегда).** Exportify → Export каждого `__ORPHANS_NN` → положить CSV в `data/Exportify/exportify_orphan_export/`.
8. **Ячейка 7.** Проверить покрытие и наличие фич. Если 100% → запускать `00_1`.

**Нет сирот?** Пропустить `00_0`, начинать с `00_1`.

## Пререквизиты

- **`src/` на месте.** Пути берутся из `src/paths.py` (`get_data_path()`, `get_project_root()`). `get_project_root()` ищет папку с `src/` и `01_data_analysis/` вверх по дереву от текущей директории — оба в репо есть.
- Extended streaming history + Account data (Spotify → Account Privacy).
- Экспорт всех плейлистов и лайков через Exportify → `data/Exportify/spotify_playlists (*)/`.
- **Только для API-режима:** Premium-аккаунт + `spotify_credentials.json` с `client_id`, `client_secret`, `redirect_uri`. `redirect_uri` должен совпадать с зарегистрированным в [Spotify Developer Dashboard](https://developer.spotify.com/dashboard). Нет Premium → использовать `'manual'`.

## Входы / Выходы

| Вход | Путь |
|---|---|
| История прослушиваний | `data/Spotify/Spotify_ Extended_Streaming_History/**/Streaming_History_Audio_*.json` |
| Account data | `data/Spotify/Spotify_Account_ Data/*.json` |
| Плейлисты Exportify | `data/Exportify/spotify_playlists (*)/*.csv` |
| Spotify-креды (API-режим) | `spotify_credentials.json` |

| Выход | Путь |
|---|---|
| `orphans_uris.csv` | `data/Exportify/orphans_to_add/` |
| `orphans_batch_NN.csv` | `data/Exportify/orphans_to_add/` |
| `created_playlists.json` | `data/Exportify/orphans_to_add/` |

## Ограничения

- **N=1** — проверено на одном профиле, числа не переносятся.
- **Premium обязателен для API-режима** — free-аккаунт вернёт 403, использовать `'manual'`.
- **Exportify — единственный ручной шаг** — API у Exportify нет.
- **Атомарности нет** — падение API посреди батча → conflict при повторном запуске, плейлист удалить вручную.
- **7 из 18 742 без `Valence`** — Exportify их не вернул; `00_1` кладёт их в `orphan_category='unknown'`.
- **Сироты зависят от момента экспорта Exportify** — порядок: экспорт → `00_0` → не менять плейлисты → `00_1`.

## Карта ячеек

| # | Секция |
|---|--------|
| 0 | Импорты, пути, создание папок |
| 0.5 | Spotify API-креды |
| 1 | Уникальные треки из истории |
| 2 | Уникальные треки из плейлистов Exportify |
| 3 | Сравнение |
| 4 | Сохранение списков сирот |
| 5 | Развилка метода (`api` / `manual`) |
| 6A | Создание плейлистов через Spotify API |
| 6B | Ручная инструкция |
| 7 | Финальная проверка |

</details>